In [16]:
import asyncio
import httpx

limits = httpx.Limits(max_keepalive_connections = 5,max_connections =10)

timeout = httpx.Timeout(10.0,connect=3.0)

client = httpx.AsyncClient(limits=limits,timeout = timeout)

In [17]:
concurrency_limits = asyncio.Semaphore(3)

async def stream_url(url: str):
    async with concurrency_limits:
        async with client.stream("GET", url) as response:
            response.raise_for_status()
            async for line in response.aiter_lines():
                if line.strip():
                    yield f"[{url[-12:]}] {line[:60]}"


In [18]:
queue = asyncio.Queue(maxsize=10)

async def producer(url:str):
    try:
        async for item in stream_url(url):
            await queue.put(item)
    except asyncio.CancelledError:
            raise

async def consumer():
    while True:
        item = await queue.get()
        if item is None:
            queue.task_done()
            break
        print(f"Processed:{item}")
        await asyncio.sleep(0.05)
        queue.task_done()

In [19]:
async def main():
    global client, queue
    if client.is_closed:
        client = httpx.AsyncClient(limits=limits, timeout=timeout)
    queue = asyncio.Queue(maxsize=10)

    urls = [
        "https://httpbin.org/stream/3",
        "https://httpbin.org/stream/2",
        "https://httpbin.org/stream/4",
    ]

    try:
        async with asyncio.timeout(8.0):
            async with asyncio.TaskGroup() as tg:
                consumer_task = tg.create_task(consumer())

                producer_tasks = [tg.create_task(producer(url)) for url in urls]

                for p in producer_tasks:
                    await p
                
                await queue.put(None)
    except* TimeoutError:
        print("Timeout Exceeded!")
    except* httpx.HTTPError as eg:
        print(f"Network Error: {eg.exceptions}")
    finally:
        await client.aclose()

await main()


Processed:[org/stream/4] {"url": "https://httpbin.org/stream/4", "args": {}, "headers
Processed:[org/stream/4] {"url": "https://httpbin.org/stream/4", "args": {}, "headers
Processed:[org/stream/4] {"url": "https://httpbin.org/stream/4", "args": {}, "headers
Processed:[org/stream/4] {"url": "https://httpbin.org/stream/4", "args": {}, "headers
Processed:[org/stream/2] {"url": "https://httpbin.org/stream/2", "args": {}, "headers
Processed:[org/stream/2] {"url": "https://httpbin.org/stream/2", "args": {}, "headers
Processed:[org/stream/3] {"url": "https://httpbin.org/stream/3", "args": {}, "headers
Processed:[org/stream/3] {"url": "https://httpbin.org/stream/3", "args": {}, "headers
Processed:[org/stream/3] {"url": "https://httpbin.org/stream/3", "args": {}, "headers
